# 5. Report

Scores every run of one experiment with aircheck-benchmark-core and writes the report. The same thing as
`aircheck-bench report` and the last step of `run_all.sh`. Replaces
[../../notebooks/collect_results.ipynb](../../notebooks/collect_results.ipynb).

Every run folder with a `run.json` is picked up automatically, including other models' runs if they are in
the same experiment folder, so nothing has to be listed here.

## Settings

In [ ]:
EXPERIMENT = "sample-benchmark"
TEST_PATH = "../../sample_data/test.parquet"    # for the Data section

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make the benchmark code importable (works from this folder or from the repo root)
BENCHMARK = next(p / rel for p in [Path.cwd(), *Path.cwd().parents] for rel in ("", "aircheck/benchmark")
                 if (p / rel / "methods" / "common.py").exists())
sys.path.insert(0, str(BENCHMARK))
AIRCHECK = BENCHMARK.parent
EXP_DIR = AIRCHECK / "output" / EXPERIMENT
print(f"Benchmark code : {BENCHMARK}")
print(f"Experiment     : {EXP_DIR}")

In [ ]:
FOLDS = sorted(EXP_DIR.glob("folds/*_cv_folds.npz"))
assert FOLDS, f"No folds in {EXP_DIR / 'folds'}; run 01_folds.ipynb first (or run_all.sh)"
FOLDS = FOLDS[0]
print(f"Folds : {FOLDS.name}")
print(f"Run   : {OUT}")

## The runs in this experiment

In [ ]:
from aircheck_benchmark_core.report import find_runs

runs = find_runs(EXP_DIR)
pd.DataFrame([{"run": r["folder"].name, "name": r["name"], "method": r["method"],
               "core": r.get("core_version"), "created": r.get("created")} for r in runs])

## Write the report

With `folds_npz` every run's fold predictions are checked to be exactly that fold's test rows, so a run on
other folds cannot slip into the table.

In [ ]:
from aircheck_benchmark_core.report import make_report

paths = make_report(EXP_DIR, out=EXP_DIR / "report.md", train_path=str(FOLDS).replace("_folds.npz", "_data.csv"),
                    test_path=TEST_PATH, folds_npz=FOLDS, title=EXPERIMENT)
for what, path in paths.items():
    print(f"{what:8s} {path}")

In [ ]:
from IPython.display import Markdown, display

display(Markdown(paths["report"].read_text()))

## Every number, as a table

In [ ]:
summary = pd.read_csv(paths["summary"])
summary.round(4)